In [65]:
# Import required libraries for API calls and environment configuration
# If needed:  pip install groq python-dotenv requests
import json  # For parsing and serializing JSON objects
import requests  # For making HTTP requests to external APIs
import os  # For accessing environment variables
from groq import Groq  # Groq API client for LLM interactions
from dotenv import load_dotenv  # For loading environment variables from .env file
from IPython.display import Markdown, display  # For displaying formatted output in Jupyter notebooks

# Load environment variables from .env file (contains API keys and sensitive data)
load_dotenv()

# Initialize the Groq client with API key from environment variables
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Specify the model to use for API calls
# The 70b model handles multi-step tool use more reliably than 8b
MODEL = "openai/gpt-oss-120b"

In [76]:
# Define utility functions for managing conversation history and tool implementations

# Custom function to add a user message to the conversation history
from datetime import datetime, timedelta  # Import for working with dates and times


def add_user_message(messages, text):
    """
    Add a user message to the conversation history.
    This is called whenever the user sends a message in the conversation.
    
    Args:
        messages (list): The conversation history list to append to
        text (str): The user's message content
    """
    # Create a message object with proper role identifier
    user_message = {
        'role': 'user',  # Identifies this as a user message
        'content': text  # The actual message text
    }
    # Add this message to the conversation history
    messages.append(user_message)

# Function to add system messages to the conversation history
def add_sys_message(messages, text):
    """
    Add a system message to the conversation history.
    System messages set the behavior and context for the AI assistant.
    They typically contain instructions on how the assistant should behave.
    
    Args:
        messages (list): The conversation history list to append to
        text (str): The system message content (e.g., role instructions)
    """
    # Create a system message object
    sys_message = {
        'role': 'system',  # Identifies this as a system/instruction message
        'content': text  # The instruction or context text
    }
    # Add this message to the conversation history
    messages.append(sys_message)


# ==================== TOOL DEFINITION ====================
# Define the actual function that the language model can call via tools

def get_current_datetime_function(date_format="%Y-%m-%d %H:%M:%S"):
    """
    Get the current datetime in the specified format.
    This function can be called by the language model via the tool-use mechanism.
    
    Args:
        date_format (str): The format string for datetime output (Python strftime format).
                          Default: "%Y-%m-%d %H:%M:%S" (produces "2026-09-14 15:30:45")
    
    Returns:
        str: Current datetime formatted according to the provided format string
    
    Raises:
        ValueError: If date_format is empty or None
    """
    # Validate that a format string was provided
    if not date_format:
        raise ValueError("date_format must be a non-empty string.")
    
    # Get current datetime and format it according to the specified format
    return datetime.now().strftime(date_format)


def add_days_function(days_to_add, current_date):
    """
    Add a specified number of days to the current date and return the new date in the specified format.
    This function can be called by the language model via the tool-use mechanism.
    
    Args:
        days_to_add (int): The number of days to add to the current date.
        current_date (datetime): The current date and time.
    
    Returns:
        str: New datetime after adding days, formatted according to the provided format string
    
    Raises:
        ValueError: If date_format is empty or None
    """
    
    # Validate that a format string was provided
    date_format = "%Y-%m-%d %H:%M:%S"  # Default format
    if not date_format:
        raise ValueError("date_format must be a non-empty string.")
    
    # Get current datetime and add days (for example, adding 5 days)
    new_date = datetime.strptime(current_date, date_format) + timedelta(days=days_to_add)
    
    # Return the new date formatted according to the specified format
    return new_date.strftime(date_format)


In [77]:
current_date = get_current_datetime_function()  # Get current date in default format
future_date = add_days_function(days_to_add=5, current_date=current_date)  # Add 5 days to the current date
print(f"Current Date: {current_date}")
print(f"Future Date (after adding 5 days): {future_date}")

Current Date: 2026-09-14 14:36:27
Future Date (after adding 5 days): 2026-09-19 14:36:27


In [78]:
# ==================== TOOL SCHEMA ====================
# Define the tools available to the language model
# This schema tells the model what functions it can call and how to call them
# The model will use this to decide when and how to invoke functions

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_datetime",
            "description": "Get the current datetime",
            "parameters": {
                "type": "object",
                "properties": {
                    "date_format": {
                        "type": "string",
                        "description": "The date and time format, e.g. %Y-%m-%d %H:%M:%S",
                        "default": "%Y-%m-%d %H:%M:%S"
                    },                
                },
                "required": [],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_days",
            "description": "Add a specified number of days to a given date",
            "parameters": {
                "type": "object",
                "properties": {
                    "days_to_add": {
                        "type": "integer",
                        "description": "The number of days to add to the current date"
                    },
                    "current_date": {
                        "type": "string",
                        "description": "The current date and time in format %Y-%m-%d %H:%M:%S"
                    }
                },
                "required": ["days_to_add", "current_date"],
            }
        }
    }
]

# ==================== FUNCTION REGISTRY ====================
# Create a dictionary that maps function names (strings) to their actual Python implementations
# When the model calls a tool by name, we use this dictionary to find and execute the real function
available_functions = {
    "get_current_datetime": get_current_datetime_function,  # Map the string name to the function object
    "add_days": add_days_function  # Map the add_days tool to its implementation
}

In [82]:
# ==================== MAIN CONVERSATION LOOP ====================
messages = []
add_sys_message(messages, "You are a helpful assistant that can use tools to answer questions.")
add_user_message(messages, "What will the exact date and time in India be after 5 days?")

print("Initial messages:")
for msg in messages:
    print(f"{msg['role'].upper()}: {msg['content']}")

# Step 1: Make the first API call
first_response = client.chat.completions.create(
    model=MODEL,
    max_tokens=2048,
    messages=messages,
    temperature=0.8,
    tool_choice='auto',
    tools=tools,
)

messages.append(first_response.choices[0].message)
finish_reason = first_response.choices[0].finish_reason

print('Starting AI conversation loop...')
if finish_reason == 'tool_calls':
    tc = first_response.choices[0].message.tool_calls[0]
    fnc_nm = tc.function.name
    function_arguments = tc.function.arguments
    tools_id = tc.id

cnt = 0

# ==================== TOOL CALLING LOOP ====================
while True:
    if finish_reason != 'tool_calls':
        print("\nFINAL ANSWER:")
        print(first_response.choices[0].message.content)
        break

    print(finish_reason, cnt)
    print(f"Tool call detected: {fnc_nm} with arguments: {function_arguments}")
    args = json.loads(function_arguments)
    result = available_functions[fnc_nm](**args)
    print(result)

    # Step 2: Add tool result to messages
    messages.append({
        "role": "tool",
        "tool_call_id": tools_id,
        "name": fnc_nm,
        "content": json.dumps(result),
    })

    # Step 3: Make next API call with tool result
    first_response = client.chat.completions.create(
        model=MODEL,
        max_tokens=2048,
        messages=messages,
        temperature=0.8,
        tool_choice='auto',
        tools=tools,
    )

    messages.append(first_response.choices[0].message)
    finish_reason = first_response.choices[0].finish_reason

    if finish_reason == 'tool_calls':
        tc = first_response.choices[0].message.tool_calls[0]
        fnc_nm = tc.function.name
        function_arguments = tc.function.arguments
        tools_id = tc.id

    cnt += 1


Initial messages:
SYSTEM: You are a helpful assistant that can use tools to answer questions.
USER: What will the exact date and time in India be after 5 days?
Starting AI conversation loop...
tool_calls 0
Tool call detected: get_current_datetime with arguments: {"date_format":"%Y-%m-%d %H:%M:%S"}
2026-09-14 14:52:44
tool_calls 1
Tool call detected: add_days with arguments: {"current_date":"2026-09-14 14:52:44","days_to_add":5}
2026-09-19 14:52:44

FINAL ANSWER:
The current UTC time is **2026‑09‑14 14:52:44**.  
Adding 5 days gives **2026‑09‑19 14:52:44 UTC**.  

India’s time zone is IST (UTC + 5 hours 30 minutes). Adding that offset:

- 14:52:44 UTC + 5 h 30 m = **20:22:44** on the same day.

**Therefore, the exact date and time in India after 5 days will be:**  

**2026‑09‑19 20:22:44 IST**.


In [69]:
# Debug cell: Display information about tool calls
# This iterates through the tool_calls and prints details about each one
# Note: 'tool_calls' variable needs to be defined from previous cell execution
# or this will cause a NameError
for t in range(0, len(tool_calls)):
    print(f"Tool called: {tool_calls[t]} with arguments: {tool_calls[t]}")

Tool called: ChatCompletionMessageToolCall(id='fc_d156610b-6093-4817-8a23-4c674556caf3', function=Function(arguments='{"date_format":"%Y-%m-%d %H:%M:%S"}', name='get_current_datetime'), type='function') with arguments: ChatCompletionMessageToolCall(id='fc_d156610b-6093-4817-8a23-4c674556caf3', function=Function(arguments='{"date_format":"%Y-%m-%d %H:%M:%S"}', name='get_current_datetime'), type='function')
